# Notebook 02 — Visualise DenPAR Annotations

Overlay bone-level polylines, CEJ/apex keypoints, and tooth masks
on sample radiographs to verify the annotation parser is working correctly.

**Run from the project root:** `opengeotrust-periosam-denpar/`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from pathlib import Path

DATA_ROOT = '../data/raw/DenPAR'
IMG_SIZE  = 512

In [ ]:
# Load samples
from src.data.parse_denpar import load_split
samples = load_split(DATA_ROOT, 'Training', max_samples=8)
print(f'Loaded {len(samples)} samples')

In [ ]:
def overlay_annotations(sample, img_size=512):
    """Return (H,W,3) BGR array with all annotations overlaid."""
    img = np.array(Image.open(sample.image_path).convert('L')
                   .resize((img_size, img_size), Image.BILINEAR))
    bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    orig_w, orig_h = Image.open(sample.image_path).size
    sx, sy = img_size / orig_w, img_size / orig_h

    # Tooth mask (green)
    if sample.mask_radio_path and sample.mask_radio_path.exists():
        m = np.array(Image.open(sample.mask_radio_path).convert('L')
                     .resize((img_size, img_size), Image.NEAREST))
        overlay = bgr.copy()
        overlay[m > 127] = [0, 180, 0]
        bgr = cv2.addWeighted(bgr, 0.7, overlay, 0.3, 0)

    # Bone-level polylines (orange)
    if sample.bone_ann and sample.bone_ann.polylines:
        for poly in sample.bone_ann.polylines:
            pts = np.array([[int(p['x']*sx), int(p['y']*sy)] for p in poly], dtype=np.int32)
            cv2.polylines(bgr, [pts], False, (30, 130, 255), 2)

    # CEJ points (cyan)
    if sample.keypoint_ann:
        for pt in sample.keypoint_ann.cej_points:
            cx, cy = int(pt['x']*sx), int(pt['y']*sy)
            cv2.circle(bgr, (cx, cy), 5, (255, 200, 0), -1)
        # Apex points (magenta)
        for pt in sample.keypoint_ann.apex_points:
            cx, cy = int(pt['x']*sx), int(pt['y']*sy)
            cv2.circle(bgr, (cx, cy), 5, (200, 0, 255), -1)

    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# Plot 4 annotated samples
n = min(4, len(samples))
fig, axes = plt.subplots(1, n, figsize=(6*n, 6))
if n == 1: axes = [axes]

legend_patches = [
    mpatches.Patch(color='#00b400', label='Tooth mask'),
    mpatches.Patch(color='#ff8200', label='Bone line'),
    mpatches.Patch(color='#ffc800', label='CEJ'),
    mpatches.Patch(color='#c800ff', label='Apex'),
]

for ax, sample in zip(axes, samples[:n]):
    annotated = overlay_annotations(sample, IMG_SIZE)
    ax.imshow(annotated)
    ax.set_title(sample.image_id[:14], fontsize=8)
    ax.axis('off')

axes[-1].legend(handles=legend_patches, loc='lower right', fontsize=8)
fig.suptitle('DenPAR Annotation Overlay', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/annotation_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Annotation coverage stats
all_samples = load_split(DATA_ROOT, 'Training')
n_total = len(all_samples)
n_radio_mask = sum(1 for s in all_samples if s.mask_radio_path and s.mask_radio_path.exists())
n_tooth_masks = sum(1 for s in all_samples if len(s.tooth_mask_paths) > 0)
n_kp = sum(1 for s in all_samples if s.keypoint_ann is not None
           and (len(s.keypoint_ann.cej_points) + len(s.keypoint_ann.apex_points)) > 0)
n_bone = sum(1 for s in all_samples if s.bone_ann is not None
             and len(s.bone_ann.polylines) > 0)

print(f'Training samples : {n_total}')
print(f'Radio mask       : {n_radio_mask}/{n_total} ({100*n_radio_mask/n_total:.1f}%)')
print(f'Tooth masks      : {n_tooth_masks}/{n_total} ({100*n_tooth_masks/n_total:.1f}%)')
print(f'Keypoints (CEJ/apex): {n_kp}/{n_total} ({100*n_kp/n_total:.1f}%)')
print(f'Bone-level lines : {n_bone}/{n_total} ({100*n_bone/n_total:.1f}%)')